# 03 — Wasserstein Clustering

An alternative to the HMM: cluster the *empirical distributions* of return windows using
the Wasserstein distance, following Horvath, Issa & Muguruza (2021).

**The motivation.** The Gaussian HMM assumes normally-distributed emissions. We showed before that the Gaussian assumption is badly violated.
WK-means is non-parametric: it makes no distributional assumption and compares full
distributions rather than their first two moments. The question is whether that flexibility
buys anything.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sys; sys.path.append('..')

from regime_utils import (wasserstein_distance_p, wasserstein_barycenter,
                          build_windows, wasserstein_kmeans, walk_forward_wasserstein)

df = pd.read_pickle("data/features.pkl")

## The Wasserstein distance


In contrast with clustering on moment features: reducing a distribution to (mean, variance)
makes two distributions with identical moments but different tail shapes **indistinguishable**.
Wasserstein compares the whole shape.

In one dimension, optimal transport has a **closed form** - sort both samples, pair the
quantiles, average the p-th powers of the gaps:

$$W_p(x,y) = \left(\frac{1}{n}\sum_i |x_{(i)} - y_{(i)}|^p\right)^{1/p}$$

Sorting makes the distance permutation-invariant. Two windows with the same
returns in a different order are the *same distribution* and should be at distance zero.
Comparing them in time order would measure sequence similarity, not distributional
similarity.

In [2]:
# Verification against scipy (which implements W1)
from scipy.stats import wasserstein_distance as scipy_w
a, b = np.random.randn(10000), np.random.randn(10000) + 3

print(f"Self-distance:     {wasserstein_distance_p(a, a):.6f}")
print(f"Shifted normals:   {wasserstein_distance_p(a, b, p=2):.4f}  (true shift = 3)")
print(f"p=1 vs scipy:      {wasserstein_distance_p(a, b, 1):.6f} / {scipy_w(a, b):.6f}")

Self-distance:     0.000000
Shifted normals:   2.9998  (true shift = 3)
p=1 vs scipy:      2.999789 / 2.999789


## WK-means

Standard k-means, but with the Wasserstein distance for assignment and the **Wasserstein
barycenter** for the centroid update. In 1D the barycenter also has a closed form: the
average of the quantile functions (sort each sample, average position-wise).

In [3]:
returns_arr = df["returns"].values
windows, end_idx = build_windows(returns_arr, window_size=21, step=1)
print(f"{len(windows)} windows of {windows.shape[1]} days")

labels, centroids = wasserstein_kmeans(windows, k=2, p=2, seed=42)

for j in range(2):
    r = windows[labels == j].flatten()
    print(f"Cluster {j}: {(labels==j).mean():5.1%} of time | "
          f"mean={r.mean():6.3f} | std={r.std():.2f} | "
          f"kurt={pd.Series(r).kurtosis():5.2f}")

4740 windows of 21 days
Cluster 0: 96.1% of time | mean= 0.055 | std=0.98 | kurt= 2.78
Cluster 1:  3.9% of time | mean=-0.329 | std=3.80 | kurt= 1.02


### K=2 fails: a crash detector, not a regime detector

With two clusters, the minority cluster captures only **~4% of the time** — and inspecting
the dates, it fires *only* in 2008, 2009, 2011 and 2020. Everything else — the 2015-16
correction, the 2018 selloff, the entire 2022 bear market — is classified as "calm".

The result is stable across seeds and unchanged with p=1, so it is not an artefact of
initialisation or of the quadratic penalty.

**Why.** Wasserstein measures *absolute magnitudes*. The transport cost between a calm
window (±1% days) and a moderate correction (±2%) is small; the cost against a crash window
(±10%) is enormous. The algorithm minimises within-cluster distance by isolating the
extremes and lumping everything else together.

A signal that fires 4% of the time is useless for exposure management.

In [4]:
dates_w = df.index[end_idx]
crisis_dates = dates_w[labels == np.argmin([windows[labels==j].std() for j in range(2)][::-1])]
crisis_dates = dates_w[labels == np.argmax([windows[labels==j].std() for j in range(2)])]
print("Years flagged as crisis (K=2):", sorted(crisis_dates.year.unique()))

# Normalised windows: strip magnitude, keep only shape
w_norm = windows / windows.std(axis=1, keepdims=True)
lab_n, _ = wasserstein_kmeans(w_norm, k=2, p=2, seed=42)
print(f"Normalised (shape only): minority cluster = {min((lab_n==0).mean(), (lab_n==1).mean()):.1%}")

Years flagged as crisis (K=2): [2008, 2009, 2011, 2020]
Normalised (shape only): minority cluster = 47.8%


### K=3 recovers a usable signal

With three clusters an intermediate state appears — roughly 73% calm / 25% moderate stress /
2% crash. That 25% is the same order of magnitude as the HMM's stressed state, and gives a
signal that actually fires often enough to manage exposure with.

In [5]:
for k in [3, 4]:
    lab_k, _ = wasserstein_kmeans(windows, k=k, p=2, seed=42)
    print(f"k={k}:")
    for j in range(k):
        r = windows[lab_k == j].flatten()
        print(f"  Cluster {j}: {(lab_k==j).mean():5.1%} | std={r.std():.2f}")

k=3:
  Cluster 0: 73.1% | std=0.74
  Cluster 1: 25.1% | std=1.64
  Cluster 2:  1.8% | std=4.81
k=4:
  Cluster 0: 55.7% | std=0.63
  Cluster 1:  9.6% | std=2.09
  Cluster 2: 33.2% | std=1.18
  Cluster 3:  1.6% | std=4.94


## Walk-forward

The clustering above is in-sample: the centroids were computed knowing the whole history.
To be comparable to the HMM it must be made causal, with the same architecture — **fit the
centroids on past data only, then assign each new day** by the distance from its trailing
window to those frozen centroids.

This fit/assign separation is what guarantees causality, and it is also the abstraction that
would let the same code run live: in a backtest the data arrives from history, live it
arrives from a feed, but the assignment logic is identical.

In [6]:
signal_wf = walk_forward_wasserstein(df, window_size=21, train_years=4, k=3, p=2)
signal_wf.to_pickle("data/signal_wass_wf.pkl")

valid = signal_wf.notna()
print(f"Days with signal: {valid.sum()}")
print(f"Time in non-calm clusters: {(signal_wf[valid] > 0).mean():.1%}")

Days with signal: 3736
Time in non-calm clusters: 33.4%
